# bottleneck-latent-projection composite — cx5: Latent->feature-map projection as a standalone nn.Module subclass

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `bottleneck-latent-projection`, `nn-module-subclass`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import torch.nn as nn
import torch.nn.functional as F
from einops.layers.torch import Rearrange

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "bottleneck-latent-projection"
DD_ATOM_IDS = ["bottleneck-latent-projection", "nn-module-subclass"]
DD_SUBTOPICS = ["Generative: Bottleneck latent projection", "PyTorch: nn.Module subclassing"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these two atoms compose

A reusable 'project a latent to a feature map' building block deserves its own `nn.Module` subclass: it factors out the `Linear + view` pair so it can be slotted into different generators/decoders without rewriting the reshape arithmetic each time.

1. **bottleneck-latent-projection** — `nn.Linear(latent_dim, C*H*W)` + reshape to `(B, C, H, W)`. The reshape lives in `forward`, the parameters live in the Linear.
2. **nn-module-subclass** — wrap the pair as `class LatentToFeatureMap(nn.Module): ...` so it has named children, a `forward`, and works with `.parameters()` / `.state_dict()`.

**Anatomy.**
```python
class LatentToFeatureMap(nn.Module):
    def __init__(self, latent_dim, channels, hw):
        super().__init__()                          # nn-module-subclass.
        self.channels, self.hw = channels, hw
        self.proj = nn.Linear(latent_dim, channels * hw * hw)   # bottleneck-latent-projection.
    def forward(self, z):
        return self.proj(z).view(z.size(0), self.channels, self.hw, self.hw)
```

**Why not `nn.Sequential(Linear, Reshape)`.** Because `.view` is not a Module, the reshape has to live in a `forward` somewhere — and the cleanest place is a tiny subclass like this one (the einops `Rearrange` route is the other clean option, exercised in cx2/cx3). Going via a subclass also makes the `channels` and `hw` configurable, and lets you swap in a `Conv2d(1, channels, 1)` later if you want a learned reshape without changing callers.

### Composite Exercise — Latent->feature-map projection as a standalone nn.Module subclass

**Atoms exercised together**: `bottleneck-latent-projection`, `nn-module-subclass`

Implement `cx5_make_latent_to_feature_map_cls()` — return a `LatentToFeatureMap` class.

Contract:
- `LatentToFeatureMap(latent_dim: int, channels: int, hw: int)`.
- `super().__init__()` first (atom: nn-module-subclass).
- Store `self.channels = channels` and `self.hw = hw`.
- `self.proj = nn.Linear(latent_dim, channels * hw * hw)` (atom: bottleneck-latent-projection). NO activation, no normalization on this module — it is JUST a learnable projection + reshape.
- `forward(self, z)` returns `self.proj(z).view(z.size(0), self.channels, self.hw, self.hw)`.

The test checks:
- Class, instance is `nn.Module`.
- Exactly one child named `proj`, of type `nn.Linear`, with correct shapes.
- `self.channels` and `self.hw` attributes are accessible (the test reads them).
- Forward: `(B, latent_dim) -> (B, channels, hw, hw)`.
- `forward(z)` numerically equals `mod.proj(z).view(B, channels, hw, hw)`.
- `model.parameters()` returns ONLY `proj.weight` and `proj.bias`.
- Output preserves the sign of negative entries (i.e. no hidden activation).

In [ ]:
def cx5_make_latent_to_feature_map_cls():
    class LatentToFeatureMap(nn.Module):
        def __init__(self, latent_dim, channels, hw):
            # Atom B (nn-module-subclass): super().__init__() FIRST so _parameters/_modules wire up.
            super().__init__()
            self.channels = channels
            self.hw = hw
            # Atom A (bottleneck-latent-projection): bare Linear to flattened feature map.
            self.proj = nn.Linear(latent_dim, channels * hw * hw)

        def forward(self, z):
            B = z.size(0)
            return self.proj(z).view(B, self.channels, self.hw, self.hw)

    return LatentToFeatureMap


<details><summary>Show solution — cx5</summary>

```python
def cx5_make_latent_to_feature_map_cls():
    class LatentToFeatureMap(nn.Module):
        def __init__(self, latent_dim, channels, hw):
            # Atom B (nn-module-subclass): super().__init__() FIRST so _parameters/_modules wire up.
            super().__init__()
            self.channels = channels
            self.hw = hw
            # Atom A (bottleneck-latent-projection): bare Linear to flattened feature map.
            self.proj = nn.Linear(latent_dim, channels * hw * hw)

        def forward(self, z):
            B = z.size(0)
            return self.proj(z).view(B, self.channels, self.hw, self.hw)

    return LatentToFeatureMap
```

The Case G grad-flow check is the strongest evidence that `super().__init__()` ran before the `self.proj = ...` line — without it, `nn.Module.__setattr__` doesn't add `proj` to `self._modules`, so `mod.parameters()` would be empty and `.backward()` would still 'work' but `proj.weight.grad` would still get populated (because the tensor itself has `requires_grad=True`). The Case E parameter-list check is what really nails the subclass discipline.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx5'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx5',
        'subtopics': ["Generative: Bottleneck latent projection", "PyTorch: nn.Module subclassing"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()